#ASG Airlines -  End-to-End Data Engineering Pipeline

This notebook covers data ingestion, quality assessment, cleaning, transformation, validation and preparation of data for Power BI reporting.

In [ ]:
import pandas as pd
import numpy as np
import os
import re
import logging

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving UseCase - Airlines.xlsx to UseCase - Airlines.xlsx


## 1. Data Ingestion

In [ ]:
file_name = "UseCase - Airlines.xlsx"

excel_file = pd.ExcelFile(file_name)

excel_file.sheet_names

['flights', 'payments', 'bookings', 'passengers']

In [ ]:
flights = pd.read_excel(file_name, sheet_name="flights")
payments = pd.read_excel(file_name, sheet_name="payments")
bookings = pd.read_excel(file_name, sheet_name="bookings")
passengers = pd.read_excel(file_name, sheet_name="passengers")

## 2. Initial Data Profiling

In [ ]:
dataset_summary = pd.DataFrame({
    "Dataset": ["Flights", "Payments", "Bookings", "Passengers"],
    "Rows": [
        flights.shape[0],
        payments.shape[0],
        bookings.shape[0],
        passengers.shape[0]
    ],
    "Columns": [
        flights.shape[1],
        payments.shape[1],
        bookings.shape[1],
        passengers.shape[1]
    ]
})

dataset_summary

,Dataset,Rows,Columns
0,Flights,1020,7
1,Payments,1000,4
2,Bookings,1000,9
3,Passengers,1039,9


In [ ]:
flights.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   flight_id       1020 non-null   object        
 1   airline         979 non-null    object        
 2   source          1020 non-null   object        
 3   destination     1020 non-null   object        
 4   departure_time  1020 non-null   datetime64[ns]
 5   arrival_time    1020 non-null   datetime64[ns]
 6   duration        1020 non-null   object        
dtypes: datetime64[ns](2), object(5)
memory usage: 55.9+ KB


In [ ]:
payments.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   payment_id      1000 non-null   object
 1   booking_id      1000 non-null   object
 2   amount          952 non-null    object
 3   payment_method  1000 non-null   object
dtypes: object(4)
memory usage: 31.4+ KB


In [ ]:
bookings.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 9 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   booking_id               1000 non-null   object        
 1   passenger_id             1000 non-null   object        
 2   flight_id                1000 non-null   object        
 3   booking_date             1000 non-null   datetime64[ns]
 4   status                   955 non-null    object        
 5   passport_number          1000 non-null   object        
 6   seat_number              1000 non-null   object        
 7   emergency_contact_name   1000 non-null   object        
 8   emergency_contact_phone  1000 non-null   object        
dtypes: datetime64[ns](1), object(8)
memory usage: 70.4+ KB


In [ ]:
passengers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1039 entries, 0 to 1038
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   passenger_id   1039 non-null   object        
 1   first_name     1039 non-null   object        
 2   last_name      1029 non-null   object        
 3   age            1039 non-null   int64         
 4   gender         1039 non-null   object        
 5   email          1039 non-null   object        
 6   phone          1039 non-null   object        
 7   aadhaar_id     1039 non-null   int64         
 8   date_of_birth  1039 non-null   datetime64[ns]
dtypes: datetime64[ns](1), int64(2), object(6)
memory usage: 73.2+ KB


In [ ]:
flights.head()

,flight_id,airline,source,destination,departure_time,arrival_time,duration
0,SJ010,SpiceJet,CCU,MAA,2026-04-20 23:38:41.701,2026-04-21 02:32:41.701,02:54:00
1,AI155,Air India,BOM,CCU,2026-04-20 23:35:41.703,2026-04-21 01:23:41.703,01:48:00
2,UK094,Vistara,BOM,CCU,2026-04-20 23:26:41.702,2026-04-21 01:11:41.702,01:45:00
3,AI245,Air India,BOM,CCU,2026-04-20 23:07:41.704,2026-04-21 01:43:41.704,02:36:00
4,AI192,Air India,MAA,BOM,2026-04-20 23:05:41.703,2026-04-21 04:04:41.703,04:59:00


In [ ]:
bookings.head()

,booking_id,passenger_id,flight_id,booking_date,status,passport_number,seat_number,emergency_contact_name,emergency_contact_phone
0,B1000,P1591,AI192,2025-06-14 11:37:36.951,CANCELLED,P1945887,3D,Isaac Bakshi,+91-6478475128
1,B1001,P1803,6F026,2025-11-02 11:37:36.951,CANCELLED,L3482012,18A,Anvi Konda,+91-6647078662
2,B1002,P1083,SJ010,2025-08-25 11:37:36.951,CANCELLED,G8507659,30C,Udant Dewan,+91-8405938220
3,B1003,P1364,AI069,2025-12-30 11:37:36.951,CONFIRMED,M0891776,33A,Harsh Chahal,+91-6264636839
4,B1004,P1885,UK003,2025-10-02 11:37:36.951,PENDING,N5742231,25C,Pahal Balay,+91-9336478266


In [ ]:
payments.head()

,payment_id,booking_id,amount,payment_method
0,PAY1000,B1116,9883.49,NETBANKING
1,PAY1001,B1738,8457.96,NETBANKING
2,PAY1002,B1873,6495.37,UPI
3,PAY1003,B1914,5079.38,NETBANKING
4,PAY1004,B1967,12518.31,CARD


In [ ]:
passengers.head()

,passenger_id,first_name,last_name,age,gender,email,phone,aadhaar_id,date_of_birth
0,P1000,Vivaan,Chatterjee,52,F,vivaan.chatterjee@gmail.com,+91-6896233790,433218196001,1974-04-08
1,P1001,Krishna,Reddy,15,M,krishna.reddy@hotmail.com,+91-6702632297,386379402654,2011-03-07
2,P1002,Myra,Naidu,72,M,myra.naidu@outlook.com,+91-6199585092,615594078161,1954-09-10
3,P1003,Myra,Mishra,61,F,myra.mishra@hotmail.com,+91-8719927151,310341316475,1965-03-12
4,P1004,Saanvi,Banerjee,21,M,saanvi.banerjee@outlook.com,+91-7819595113,419283276483,2005-11-11


## 3. Data Quality Assessment

In [ ]:
missing_values = pd.DataFrame({
    "Flights": flights.isnull().sum(),
    "Payments": payments.isnull().sum(),
    "Bookings": bookings.isnull().sum(),
    "Passengers": passengers.isnull().sum()
})

missing_values

,Flights,Payments,Bookings,Passengers
aadhaar_id,NaN,NaN,NaN,0.0
age,NaN,NaN,NaN,0.0
airline,41.0,NaN,NaN,NaN
amount,NaN,48.0,NaN,NaN
arrival_time,0.0,NaN,NaN,NaN
booking_date,NaN,NaN,0.0,NaN
booking_id,NaN,0.0,0.0,NaN
date_of_birth,NaN,NaN,NaN,0.0
departure_time,0.0,NaN,NaN,NaN
destination,0.0,NaN,NaN,NaN


In [ ]:
duplicate_summary = pd.DataFrame({
    "Dataset": ["Flights", "Payments", "Bookings", "Passengers"],
    "Duplicate_Rows": [
        flights.duplicated().sum(),
        payments.duplicated().sum(),
        bookings.duplicated().sum(),
        passengers.duplicated().sum()
    ]
})

duplicate_summary

,Dataset,Duplicate_Rows
0,Flights,15
1,Payments,0
2,Bookings,0
3,Passengers,0


### Flight Field Review

In [ ]:
flights["airline"].value_counts(dropna=False)

,count
airline,
IndiGo,249
SpiceJet,240
Air India,236
Vistara,223
NaN,41
UNKNOWN,31


In [ ]:
flights["source"].value_counts(dropna=False)

,count
source,
BOM,207
HYD,180
CCU,174
DEL,163
MAA,157
BLR,139


In [ ]:
flights["destination"].value_counts(dropna=False)

,count
destination,
DEL,200
CCU,188
BOM,174
BLR,165
MAA,151
HYD,142


In [ ]:
flights["flight_id"].value_counts(dropna=False).head(20)

,count
flight_id,
UK049,2
AI043,2
AI070,2
SJ037,2
SJ118,2
SJ146,2
SJ142,2
AI242,2
UK013,2


### Time Field Review

In [ ]:
flights["departure_time"].head(20)

,departure_time
0,2026-04-20 23:38:41.701
1,2026-04-20 23:35:41.703
2,2026-04-20 23:26:41.702
3,2026-04-20 23:07:41.704
4,2026-04-20 23:05:41.703
5,2026-04-20 23:05:41.703
6,2026-04-20 23:04:41.703
7,2026-04-20 23:03:41.702
8,2026-04-20 23:02:41.701
9,2026-04-20 22:56:41.703


In [ ]:
flights["arrival_time"].head(20)

,arrival_time
0,2026-04-21 02:32:41.701
1,2026-04-21 01:23:41.703
2,2026-04-21 01:11:41.702
3,2026-04-21 01:43:41.704
4,2026-04-21 04:04:41.703
5,2026-04-21 01:24:41.703
6,2026-04-21 00:47:41.703
7,2026-04-21 00:35:41.702
8,2026-04-20 23:57:41.701
9,2026-04-20 23:37:41.703


In [ ]:
flights["departure_time"].astype(str).value_counts().head(20)

,count
departure_time,
2026-04-17 17:06:41.702,2
2026-04-20 20:54:41.703,2
2026-04-17 19:47:41.701,2
2026-04-17 19:14:41.701,2
2026-04-17 18:24:41.701,2
2026-04-17 21:05:41.702,2
2026-04-17 21:18:41.703,2
2026-04-18 01:27:41.702,2
2026-04-18 06:42:41.703,2


In [ ]:
flights["arrival_time"].astype(str).value_counts().head(20)

,count
arrival_time,
2026-04-19 02:33:41.702,3
2026-04-18 15:48:41.701,2
2026-04-20 15:21:41.703,2
2026-04-19 01:25:41.701,2
2026-04-17 20:32:41.701,2
2026-04-19 23:08:41.703,2
2026-04-20 21:00:41.702,2
2026-04-18 19:42:41.701,2
2026-04-20 00:33:41.702,2


In [ ]:
flights["duration"].head(20)

,duration
0,02:54:00
1,01:48:00
2,01:45:00
3,02:36:00
4,04:59:00
5,02:19:00
6,01:43:00
7,01:32:00
8,00:55:00
9,00:41:00


In [ ]:
flights["duration"].dtype

dtype('O')

In [ ]:
flights["duration"].describe()

,duration
count,1020
unique,270
top,02:49:00
freq,11


## 4. Dataset Relationships

In [ ]:
print("Flight IDs:", flights["flight_id"].nunique())
print("Booking Flight IDs:", bookings["flight_id"].nunique())
print("Booking IDs:", bookings["booking_id"].nunique())
print("Payment Booking IDs:", payments["booking_id"].nunique())
print("Passenger IDs:", passengers["passenger_id"].nunique())
print("Booking Passenger IDs:", bookings["passenger_id"].nunique())

Flight IDs: 1004
Booking Flight IDs: 984
Booking IDs: 1000
Payment Booking IDs: 637
Passenger IDs: 1000
Booking Passenger IDs: 636


## 5. PII Identification

In [ ]:
print("Passenger dataset columns:")
print(passengers.columns.tolist())

print("\nBooking dataset columns:")
print(bookings.columns.tolist())

Passenger dataset columns:
['passenger_id', 'first_name', 'last_name', 'age', 'gender', 'email', 'phone', 'aadhaar_id', 'date_of_birth']

Booking dataset columns:
['booking_id', 'passenger_id', 'flight_id', 'booking_date', 'status', 'passport_number', 'seat_number', 'emergency_contact_name', 'emergency_contact_phone']


## 6. Quality Summary

In [ ]:
quality_summary = pd.DataFrame({
    "Dataset": ["Flights", "Payments", "Bookings", "Passengers"],
    "Rows": [
        len(flights),
        len(payments),
        len(bookings),
        len(passengers)
    ],
    "Missing_Values": [
        flights.isnull().sum().sum(),
        payments.isnull().sum().sum(),
        bookings.isnull().sum().sum(),
        passengers.isnull().sum().sum()
    ],
    "Duplicate_Rows": [
        flights.duplicated().sum(),
        payments.duplicated().sum(),
        bookings.duplicated().sum(),
        passengers.duplicated().sum()
    ]
})

quality_summary

,Dataset,Rows,Missing_Values,Duplicate_Rows
0,Flights,1020,41,15
1,Payments,1000,48,0
2,Bookings,1000,45,0
3,Passengers,1039,10,0


## 7. Data Cleaning

### 7.1 Duplicate Records

In [ ]:
flights_clean = flights.copy()

In [ ]:
flights_clean = flights_clean.drop_duplicates().reset_index(drop=True)

In [ ]:
flights_clean.duplicated().sum()

np.int64(0)

In [ ]:
print("Rows before cleaning:", len(flights))
print("Rows after duplicate removal:", len(flights_clean))
print("Duplicate rows remaining:", flights_clean.duplicated().sum())

Rows before cleaning: 1020
Rows after duplicate removal: 1005
Duplicate rows remaining: 0


The flight dataset contained 15 exact duplicate records. After verifying that the duplicated rows contained identical flight details, one copy of each duplicate was retained and the redundant records were removed.

### 7.2 Flight ID Consistency

In [ ]:
conflicting_flight_ids = (
    flights_clean.groupby("flight_id")
    .filter(lambda x: len(x) > 1 and len(x.drop_duplicates()) > 1)
)

conflicting_flight_ids

,flight_id,airline,source,destination,departure_time,arrival_time,duration
250,6F250,UNKNOWN,DEL,BLR,2026-04-20 03:26:41.701,2026-04-20 07:30:41.701,04:04:00
267,6F250,UNKNOWN,CCU,BLR,2026-04-20 02:23:41.702,2026-04-20 02:56:41.702,00:33:00


In [ ]:
flights_clean["id_conflict_flag"] = flights_clean["flight_id"].duplicated(keep=False)

In [ ]:
flights_clean[flights_clean["id_conflict_flag"]]

,flight_id,airline,source,destination,departure_time,arrival_time,duration,id_conflict_flag
250,6F250,UNKNOWN,DEL,BLR,2026-04-20 03:26:41.701,2026-04-20 07:30:41.701,04:04:00,True
267,6F250,UNKNOWN,CCU,BLR,2026-04-20 02:23:41.702,2026-04-20 02:56:41.702,00:33:00,True


In [ ]:
flights_clean["id_conflict_flag"].value_counts()

,count
id_conflict_flag,
False,1003
True,2


### 7.3 Missing Airline Values

In [ ]:
missing_airline = flights_clean[flights_clean["airline"].isnull()]

missing_airline

,flight_id,airline,source,destination,departure_time,arrival_time,duration,id_conflict_flag
57,UK209,NaN,MAA,BLR,2026-04-20 19:02:41.704,2026-04-20 21:47:41.704,02:45:00,False
63,6F270,NaN,HYD,MAA,2026-04-20 18:14:41.701,2026-04-20 19:02:41.701,00:48:00,False
88,UK224,NaN,CCU,MAA,2026-04-20 16:25:41.704,2026-04-20 20:24:41.704,03:59:00,False
116,6F261,NaN,CCU,DEL,2026-04-20 14:31:41.703,2026-04-20 17:29:41.703,02:58:00,False
118,6F254,NaN,MAA,BLR,2026-04-20 14:26:41.703,2026-04-20 18:01:41.703,03:35:00,False
122,6F259,NaN,BOM,DEL,2026-04-20 14:10:41.703,2026-04-20 15:21:41.703,01:11:00,False
131,UK141,NaN,HYD,MAA,2026-04-20 13:38:41.703,2026-04-20 15:59:41.703,02:21:00,False
147,6F255,NaN,DEL,BOM,2026-04-20 11:59:41.703,2026-04-20 15:33:41.703,03:34:00,False
215,AI047,NaN,CCU,MAA,2026-04-20 06:36:41.701,2026-04-20 07:23:41.701,00:47:00,False
219,AI128,NaN,BOM,CCU,2026-04-20 06:07:41.702,2026-04-20 10:03:41.702,03:56:00,False


In [ ]:
missing_airline.shape

(39, 8)

In [ ]:
missing_airline[["flight_id", "source", "destination"]]

,flight_id,source,destination
57,UK209,MAA,BLR
63,6F270,HYD,MAA
88,UK224,CCU,MAA
116,6F261,CCU,DEL
118,6F254,MAA,BLR
122,6F259,BOM,DEL
131,UK141,HYD,MAA
147,6F255,DEL,BOM
215,AI047,CCU,MAA
219,AI128,BOM,CCU


In [ ]:
missing_airline["flight_id"].astype(str).str[:2].value_counts()

,count
flight_id,
AI,14
6F,12
UK,8
SJ,5


In [ ]:
airline_prefix_check = flights_clean.copy()

airline_prefix_check["flight_prefix"] = (
    airline_prefix_check["flight_id"]
    .astype(str)
    .str.extract(r"^([A-Za-z0-9]+?)(?=\d+$)")[0]
)

airline_prefix_check.groupby(
    ["flight_prefix", "airline"],
    dropna=False
).size().sort_values(ascending=False)

flight_prefix  airline  
6F             IndiGo       249
SJ             SpiceJet     236
AI             Air India    233
UK             Vistara      218
AI             NaN           14
6F             UNKNOWN       12
               NaN           12
AI             UNKNOWN        8
UK             NaN            8
SJ             UNKNOWN        6
               NaN            5
UK             UNKNOWN        4
dtype: int64

In [ ]:
flights_clean["airline"].value_counts(dropna=False)

,count
airline,
IndiGo,249
SpiceJet,236
Air India,233
Vistara,218
NaN,39
UNKNOWN,30


In [ ]:
prefix_airline_summary = (
    airline_prefix_check
    .groupby("flight_prefix")["airline"]
    .value_counts(dropna=False)
    .unstack(fill_value=0)
)

prefix_airline_summary

airline,Air India,IndiGo,SpiceJet,UNKNOWN,Vistara,NaN
flight_prefix,,,,,,
6F,0,249,0,12,0,12
AI,233,0,0,8,0,14
SJ,0,0,236,6,0,5
UK,0,0,0,4,218,8


In [ ]:
airline_mapping = {
    "AI": "Air India",
    "6F": "IndiGo",
    "SJ": "SpiceJet",
    "UK": "Vistara"
}

In [ ]:
flight_prefix = flights_clean["flight_id"].astype(str).str[:2]

airline_missing_mask = (
    flights_clean["airline"].isnull() |
    flights_clean["airline"].str.upper().eq("UNKNOWN")
)

flights_clean.loc[airline_missing_mask, "airline"] = (
    flight_prefix[airline_missing_mask].map(airline_mapping)
)

In [ ]:
flights_clean["airline"].value_counts(dropna=False)

,count
airline,
IndiGo,273
Air India,255
SpiceJet,247
Vistara,230


In [ ]:
flights_clean["airline"].isnull().sum()

np.int64(0)

In [ ]:
flights_clean["airline"].str.upper().eq("UNKNOWN").sum()

np.int64(0)

In [ ]:
flights_clean["airline"].unique()

array(['SpiceJet', 'Air India', 'Vistara', 'IndiGo'], dtype=object)

The missing or blank airline values were standardized based on the flight ID prefix. The consistency in the mapping between prefix and airline was present in the original data, which helped in recovering the affected records without deleting other flight records.

### 7.4 Time Standardization

In [ ]:
flights_clean[["departure_time", "arrival_time"]].dtypes

,0
departure_time,datetime64[ns]
arrival_time,datetime64[ns]


In [ ]:
flights_clean["calculated_duration"] = (
    flights_clean["arrival_time"] -
    flights_clean["departure_time"]
)

flights_clean[
    ["flight_id", "departure_time", "arrival_time", "calculated_duration"]
].head()

,flight_id,departure_time,arrival_time,calculated_duration
0,SJ010,2026-04-20 23:38:41.701,2026-04-21 02:32:41.701,0 days 02:54:00
1,AI155,2026-04-20 23:35:41.703,2026-04-21 01:23:41.703,0 days 01:48:00
2,UK094,2026-04-20 23:26:41.702,2026-04-21 01:11:41.702,0 days 01:45:00
3,AI245,2026-04-20 23:07:41.704,2026-04-21 01:43:41.704,0 days 02:36:00
4,AI192,2026-04-20 23:05:41.703,2026-04-21 04:04:41.703,0 days 04:59:00


In [ ]:
overnight_flights = flights_clean[
    flights_clean["arrival_time"].dt.date >
    flights_clean["departure_time"].dt.date
]

print("Overnight flights:", len(overnight_flights))

Overnight flights: 122


In [ ]:
invalid_time_order = flights_clean[
    flights_clean["arrival_time"] <
    flights_clean["departure_time"]
]

invalid_time_order[
    ["flight_id", "departure_time", "arrival_time", "duration"]
]

,flight_id,departure_time,arrival_time,duration
351,SJ192,2026-04-19 18:45:42,2026-04-18 23:45:42,1899-12-29 05:00:00


In [ ]:
flights_clean["calculated_duration"] = (
    flights_clean["arrival_time"] -
    flights_clean["departure_time"]
)

In [ ]:
flights_clean["duration_timedelta"] = flights_clean["duration"].apply(
    lambda x: pd.Timedelta(
        hours=x.hour,
        minutes=x.minute,
        seconds=x.second
    ) if pd.notna(x) else pd.NaT
)

In [ ]:
duration_mismatches = flights_clean[
    flights_clean["duration_timedelta"] !=
    flights_clean["calculated_duration"].dt.round("min")
]

print("Duration mismatches:", len(duration_mismatches))

Duration mismatches: 1


In [ ]:
print("Total flights:", len(flights_clean))
print("Overnight flights:", len(overnight_flights))
print("Duration mismatches:", len(duration_mismatches))

Total flights: 1005
Overnight flights: 122
Duration mismatches: 1


In [ ]:
duration_mismatches[
    [
        "flight_id",
        "departure_time",
        "arrival_time",
        "duration",
        "duration_timedelta",
        "calculated_duration"
    ]
]

,flight_id,departure_time,arrival_time,duration,duration_timedelta,calculated_duration
351,SJ192,2026-04-19 18:45:42,2026-04-18 23:45:42,1899-12-29 05:00:00,0 days 05:00:00,-1 days +05:00:00


In [ ]:
flights_clean.loc[
    flights_clean["flight_id"] == "SJ192",
    "arrival_time"
] = (
    flights_clean.loc[
        flights_clean["flight_id"] == "SJ192",
        "arrival_time"
    ] + pd.Timedelta(days=1)
)

In [ ]:
flights_clean["calculated_duration"] = (
    flights_clean["arrival_time"] -
    flights_clean["departure_time"]
)

In [ ]:
duration_mismatches = flights_clean[
    flights_clean["duration_timedelta"] !=
    flights_clean["calculated_duration"].dt.round("min")
]

print("Duration mismatches:", len(duration_mismatches))

Duration mismatches: 0


###7.5 PII Protection

In [ ]:
passengers_clean = passengers.copy()

print("Passenger columns:")
print(passengers_clean.columns.tolist())

Passenger columns:
['passenger_id', 'first_name', 'last_name', 'age', 'gender', 'email', 'phone', 'aadhaar_id', 'date_of_birth']


In [ ]:
import hashlib

def hash_pii(value):
    if pd.isna(value):
        return value
    return hashlib.sha256(str(value).encode()).hexdigest()

In [ ]:
pii_columns = [
    "first_name",
    "last_name",
    "email",
    "phone",
    "aadhaar_id",
    "date_of_birth"
]

for col in pii_columns:
    passengers_clean[col] = passengers_clean[col].apply(hash_pii)

In [ ]:
print("Protected passenger data:")
display(passengers_clean[pii_columns].head())

Protected passenger data:


,first_name,last_name,email,phone,aadhaar_id,date_of_birth
0,3ec26520ba7deef4bc8b4164ff0067ec1b7a01fbc62d1d...,c43884b497e2850780ead34ae9d7ce4daa804222cfd5b6...,86ebf4745fa93e45ba826eff0d227a18f8c65329d6a9ac...,d254e34512e7e456e1b1713f7584492aaa0e014d072f36...,99466baa9b8fe69a6f31db34c87d50f0e9a488eda788f3...,3d24528a8612aacf11a507aff4923b681040daf8c3456e...
1,4d9b474d4634b01eabb38aff399935ddfb56899da3bf89...,2ae0f064b95fc445516cf0d5b91dd7eccf4ba6e0c28d0d...,17744128240944e417f265a838038b2ee3f6573e74d15d...,c1c4f343ca8c0075d18e63d34b2dfa1ea896fb1a8ebb37...,6d05ccbeb5011dd59948b882fff86e1a6f7fc0cf41f659...,95d0dc89eae58481eed892ce9ca3aa92c1ba3acb5a0eb0...
2,fd9199745082e09dccdc27a82fa8c707627a4f215cbb4d...,d42c60721b8cbcbfb28df944aa63d6ed1710361897850d...,884533acfd5cfbe6787f3d61f07ed253a19430b8d412be...,bbb0f22af3e01021a7f0bbbc1d659f0dafb36610f77695...,3e24ac79e2c956797c22051676644d26204cc4de0f203e...,06c760809a95d3e2ad9517a8263659c9a3ad8f966b5745...
3,fd9199745082e09dccdc27a82fa8c707627a4f215cbb4d...,6c5ad499275b19ecd774dd51337bac3444f792e48e31e4...,844b61f750119ea14552675bc6a98c16274d32fa6d9f49...,a0ae77b555b82560695b5a1133bb5a6cc30866c19fbc14...,4e880723014ee24b39f0213150496f3efb9faa3f62d335...,d6c749f8f47f64974ed93b32c97095613ca1c460cab383...
4,9232592eb181257978d267d15add3f4169113a4b65d286...,5887b467e3a50aec1fa4e359743042f6940325cc51c147...,3d941c6227ef310fc68146f452ce49dd5c5539e326eacb...,7428193446b16270dae5c09793ea0d3391d0fef3f38f87...,3ca09c31f18da71ad2bc39cb3cbb387c38a9cd5ad37239...,83c8b8fc21e42ed18ce2897ee383308841e6f73fb6ee49...


In [ ]:
print("PII Protection Validation")
for col in pii_columns:
    print(col, ":", passengers_clean[col].notna().sum(), "non-null values")

PII Protection Validation
first_name : 1039 non-null values
last_name : 1029 non-null values
email : 1039 non-null values
phone : 1039 non-null values
aadhaar_id : 1039 non-null values
date_of_birth : 1039 non-null values


###7.6 Final Validation

In [ ]:
print("FINAL FLIGHT DATA VALIDATION")
print("-" * 40)

print("Total flight records:", len(flights_clean))
print("Duplicate records:", flights_clean.duplicated().sum())
print("Missing airline values:", flights_clean["airline"].isna().sum())
print("UNKNOWN airline values:", flights_clean["airline"].str.upper().eq("UNKNOWN").sum())
print("Conflicting flight IDs:", flights_clean["id_conflict_flag"].sum())
print("Invalid time order:", (flights_clean["arrival_time"] < flights_clean["departure_time"]).sum())
print("Duration mismatches:", len(duration_mismatches))

FINAL FLIGHT DATA VALIDATION
----------------------------------------
Total flight records: 1005
Duplicate records: 0
Missing airline values: 0
UNKNOWN airline values: 0
Conflicting flight IDs: 2
Invalid time order: 0
Duration mismatches: 0


## 7.7 KPI & Aggregation

In [ ]:
flights_clean["duration_minutes"] = (
    flights_clean["calculated_duration"].dt.total_seconds() / 60
)

print(flights_clean["duration_minutes"].describe())

count    1005.000000
mean      164.620034
std        77.463864
min        30.000000
25%        99.000000
50%       166.000000
75%       234.000000
max       300.000000
Name: duration_minutes, dtype: float64


### KPI 1: Average Flight Duration

In [ ]:
average_flight_duration = flights_clean["duration_minutes"].mean()

print("Average Flight Duration:", round(average_flight_duration, 2), "minutes")

Average Flight Duration: 164.62 minutes


### KPI 2: Route-wise Traffic

In [ ]:
route_traffic = (
    flights_clean
    .groupby(["source", "destination"])
    .size()
    .reset_index(name="flight_count")
    .sort_values("flight_count", ascending=False)
)

display(route_traffic.head(10))

,source,destination,flight_count
6,BOM,CCU,90
12,CCU,DEL,72
25,MAA,BLR,65
0,BLR,BOM,60
24,HYD,MAA,57
18,DEL,HYD,54
23,HYD,DEL,42
7,BOM,DEL,39
11,CCU,BOM,33
15,DEL,BLR,29


### KPI 3: Delays / Anomalies

In [ ]:
anomaly_summary = pd.DataFrame({
    "anomaly_type": [
        "Flight ID Conflict",
        "Overnight Flight"
    ],
    "count": [
        flights_clean["id_conflict_flag"].sum(),
        len(overnight_flights)
    ]
})

display(anomaly_summary)

,anomaly_type,count
0,Flight ID Conflict,2
1,Overnight Flight,122


### KPI 4: Distribution of Flights by Airline

In [ ]:
airline_distribution = (
    flights_clean["airline"]
    .value_counts()
    .reset_index()
)

airline_distribution.columns = ["airline", "flight_count"]

display(airline_distribution)

,airline,flight_count
0,IndiGo,273
1,Air India,255
2,SpiceJet,247
3,Vistara,230


## 7.8 Export Cleaned & Aggregated Data

In [ ]:
flights_clean.to_csv("cleaned_flights.csv", index=False)
passengers_clean.to_csv("protected_passengers.csv", index=False)
route_traffic.to_csv("route_traffic.csv", index=False)
anomaly_summary.to_csv("anomaly_summary.csv", index=False)
airline_distribution.to_csv("airline_distribution.csv", index=False)

print("All datasets exported successfully.")

All datasets exported successfully.


In [ ]:
import os

exported_files = [
    "cleaned_flights.csv",
    "protected_passengers.csv",
    "route_traffic.csv",
    "anomaly_summary.csv",
    "airline_distribution.csv"
]

for file in exported_files:
    print(file, "->", os.path.getsize(file), "bytes")

cleaned_flights.csv -> 124684 bytes
protected_passengers.csv -> 415955 bytes
route_traffic.csv -> 362 bytes
anomaly_summary.csv -> 61 bytes
airline_distribution.csv -> 71 bytes
